<div dir="rtl">

# جلسه ۲۰: پروژه‌ی ۲ — شبیه‌سازی صف یک شعبه‌ی بانک
## Project 2: Simulating a Bank Branch Queue

**بخش ۲: پایتون پیشرفته** — دوره‌ی پایتون برای علوم داده

مدرس: دکتر حسین حق‌بین

</div>

<div dir="rtl">

## فهرست

1. صورت مسئله
2. داده
3. الزامات برنامه
4. محدودیت‌ها
5. آنچه تحویل می‌دهید
6. دفاع
7. معیارهای ارزیابی

</div>

<div dir="rtl">

## ۱. صورت مسئله

مدیر یک شعبه‌ی بانک از شکایت مشتری‌ها درباره‌ی **صف‌های طولانی** نگران است. شعبه یک سیستم نوبت‌دهی دارد: هر مشتری هنگام ورود یک شماره (ticket) می‌گیرد و منتظر می‌ماند تا یکی از باجه‌ها آزاد شود. سامانه‌ی نوبت‌دهی برای هر مشتری زمان ورود، زمان شروع و پایان خدمت، باجه و نوع خدمت را ثبت کرده است.

مدیر خروجی دو هفته از این سامانه را در اختیار شما گذاشته است (بخش ۲) و از شما خواسته است:

- داده را **پاک‌سازی** کنید و بگویید با هر ایراد چه کرده‌اید،
- **وضعیت فعلی** شعبه را توصیف کنید: مشتری‌ها چقدر منتظر می‌مانند، شلوغ‌ترین ساعت‌ها کدام‌اند، باجه‌ها چقدر کار می‌کنند و چه چیز غیرعادی‌ای در داده دیده می‌شود،
- یک **شبیه‌ساز** شعبه بسازید که با آن بتوان بدون آزمایش روی مشتری‌های واقعی، سؤال «چه می‌شود اگر...؟» را پاسخ داد،
- با شبیه‌ساز چند **سیاست** را مقایسه کنید و به **پرسش‌های مدیر** (الزام ۷) پاسخ دهید.

همه‌ی نتایج را در یک **گزارش** ارائه می‌کنید که مدیر، بدون دانستن برنامه‌نویسی، بتواند بر اساس آن تصمیم بگیرد.

</div>

<p align="center"><img src="images/s20_pipeline.png" width="900" alt="s20_pipeline.png"></p>

<p align="center" dir="rtl"><i>مراحل اصلی برنامه و جلسه‌هایی که ابزار هر مرحله را در آن‌ها یاد گرفته‌اید.</i></p>

<div dir="rtl">

### قواعد شعبه و شبیه‌سازی

| مفهوم | تعریف |
|---|---|
| ساعت کار | شنبه تا چهارشنبه، <code dir="ltr">08:00</code> تا <code dir="ltr">14:00</code>؛ در ساعت <code dir="ltr">14:00</code> در بسته می‌شود، اما مشتری‌هایی که داخل شعبه‌اند تا آخر خدمت می‌گیرند |
| زمان انتظار | زمان شروع خدمت منهای زمان ورود (دقیقه) |
| زمان خدمت | زمان پایان خدمت منهای زمان شروع خدمت (دقیقه) |
| سیاست فعلی شعبه | **یک صف مشترک** (نوبت‌دهی)؛ نفر اول صف به اولین باجه‌ی آزاد می‌رود |
| واحد زمان شبیه‌سازی | **دقیقه**؛ در هر دقیقه از ساعت <code dir="ltr">h</code>، یک مشتری با احتمال <code dir="ltr">λ&#95;h / 60</code> وارد می‌شود، که <code dir="ltr">λ&#95;h</code> میانگین تعداد ورود در آن ساعت در داده است (با <code dir="ltr">random.random()</code>) |
| نوع خدمت مشتری شبیه‌سازی‌شده | به‌طور تصادفی، با همان نسبت‌هایی که در داده دیده می‌شود (<code dir="ltr">random.choices</code> با وزن) |
| زمان خدمت مشتری شبیه‌سازی‌شده | به‌طور تصادفی از میان زمان‌های خدمت **واقعیِ** همان نوع خدمت در داده (<code dir="ltr">random.choice</code>) |
| صف جدا برای هر باجه (<code dir="ltr">multi</code>) | هر مشتری هنگام ورود به **کوتاه‌ترین** صف می‌رود و دیگر صفش را عوض نمی‌کند |
| باجه‌ی سریع (<code dir="ltr">express</code>) | باجه‌ی شماره‌ی ۱ فقط به خدمت‌های کوتاه (<code dir="ltr">deposit</code> و <code dir="ltr">withdrawal</code>) رسیدگی می‌کند؛ باقی باجه‌ها از هر دو صف، به ترتیب زمان ورود، مشتری می‌پذیرند |

</div>

<div dir="rtl">

## ۲. داده

داده یک فایل CSV به نام <code dir="ltr">bank&#95;log.csv</code> است. آن را از
[این پیوند](https://haghbinh.github.io/Python-Course/Notebooks/data/bank_log.csv)
دانلود کنید و در پوشه‌ای به نام <code dir="ltr">data</code> کنار نوت‌بوک خود بگذارید. (اگر کل مخزن را با **Download ZIP** گرفته‌اید، فایل همین حالا در <code dir="ltr">Notebooks/data</code> هست.) فایل را **دستی ویرایش نکنید**: پیدا کردن و رفع ایرادها باید با **کد** انجام شود.

سطر اول عنوان ستون‌هاست و هر سطر بعدی یک مشتری است:

<div dir="ltr">

```text
ticket,date,arrival,service_start,service_end,teller,service_type
00002,2026-11-14,08:02:31,08:02:31,08:05:36,T1,withdrawal
```

</div>

| ستون | معنی |
|---|---|
| <code dir="ltr">ticket</code> | شماره‌ی نوبت |
| <code dir="ltr">date</code> | تاریخ (<code dir="ltr">YYYY-MM-DD</code>) |
| <code dir="ltr">arrival</code> | زمان ورود و گرفتن نوبت |
| <code dir="ltr">service&#95;start</code> / <code dir="ltr">service&#95;end</code> | زمان شروع و پایان خدمت؛ **خالی** یعنی مشتری قبل از رسیدن نوبتش شعبه را **ترک کرده** است |
| <code dir="ltr">teller</code> | باجه (<code dir="ltr">T1</code>، <code dir="ltr">T2</code>، <code dir="ltr">T3</code>)؛ برای مشتری‌ای که رفته خالی است |
| <code dir="ltr">service&#95;type</code> | نوع خدمت: <code dir="ltr">deposit</code> (واریز)، <code dir="ltr">withdrawal</code> (برداشت)، <code dir="ltr">transfer</code> (انتقال)، <code dir="ltr">loan</code> (وام)، <code dir="ltr">account&#95;opening</code> (افتتاح حساب) |

داده ساختگی است، اما ایرادهای آن (سطر تکراری، قالب‌های ناهمسان، زمان‌های ناممکن و ...) از نوع ایرادهای داده‌های واقعی است. سلول زیر فقط بررسی می‌کند که فایل در جای درست است:

</div>

In [ ]:
from pathlib import Path

DATA = Path("data/bank_log.csv")
print(DATA.exists())

<div dir="rtl">

## ۳. الزامات برنامه

طراحی کلاس‌ها و توابع با خودتان است، اما **سه تابع زیر باید دقیقاً با همین امضا وجود داشته باشند**، چون در جلسه‌ی دفاع با داده و پارامترهای جدید آزمایش می‌شوند:

<div dir="ltr">

```python
records, rejected = clean_log(path)
# records:  cleaned records (your own structure: dicts, tuples or objects)
# rejected: list of (line_number, line, reason) tuples

model = fit_model(records)
# everything the simulator needs, estimated from the data

result = simulate(model, policy="single", n_tellers=3, n_days=200, seed=0)
# policy: "single", "multi" or "express"
# result: dict with keys "mean_wait", "p90_wait", "share_over_15",
#         "max_queue", "utilization"
```

</div>

1. **پاک‌سازی:** <code dir="ltr">clean&#95;log</code> باید ایرادهای داده را پیدا کند. سطری که با اطمینان قابل اصلاح است اصلاح شود و سطری که قابل اصلاح نیست با شماره‌ی سطر و **دلیل** در <code dir="ltr">rejected</code> ثبت شود؛ هیچ سطری نباید بی‌صدا حذف شود. برای خطاهای داده یک **استثنای سفارشی** (مثلاً <code dir="ltr">DataError</code>) تعریف کنید و با <code dir="ltr">try</code> / <code dir="ltr">except</code> مدیریتش کنید. مشتری‌هایی که شعبه را ترک کرده‌اند **خطا نیستند**؛ تصمیم بگیرید در تحلیل با آن‌ها چه می‌کنید. در گزارش، برای هر نوع ایراد بگویید چند بار دیدید و چه تصمیمی گرفتید و چرا.
2. **وضعیت فعلی شعبه:**
   - زمان انتظار: میانگین، میانه، صدک ۹۰ و درصد مشتری‌هایی که بیش از ۱۵ دقیقه منتظر مانده‌اند؛ برای کل داده، هر روز و هر ساعت.
   - تعداد ورود مشتری در هر ساعت روز (میانگین روی روزها).
   - زمان خدمت هر نوع خدمت و هر باجه.
   - تعداد مشتری‌هایی که شعبه را ترک کرده‌اند و زمان آن.
3. **موارد غیرعادی:** هر روز، باجه یا الگوی غیرعادی را که در داده می‌بینید گزارش و با عدد مستند کنید و برای آن توضیحی پیشنهاد دهید.
4. **مدل:** <code dir="ltr">fit&#95;model</code> باید از داده‌ی پاک‌شده نرخ ورود هر ساعت، نسبت نوع خدمت‌ها و فهرست زمان‌های خدمت هر نوع را استخراج کند. اگر تصمیم می‌گیرید بخشی از داده را برای ساختن مدل کنار بگذارید، دلیلش را بگویید.
5. **شبیه‌ساز شیءگرا:** دست‌کم کلاس‌های <code dir="ltr">Customer</code>، <code dir="ltr">Teller</code> و یک کلاس پایه‌ی <code dir="ltr">Bank</code> بسازید و هر سیاست صف (<code dir="ltr">single</code>، <code dir="ltr">multi</code>، <code dir="ltr">express</code>) یک **زیرکلاس** آن باشد که فقط رفتار متفاوتش را بازنویسی می‌کند (وراثت و چندریختی، جلسه‌های ۱۸ و ۱۹). از <code dir="ltr">deque</code> برای صف‌ها استفاده کنید. نتیجه‌ی <code dir="ltr">simulate</code> با <code dir="ltr">seed</code> یکسان باید **عیناً تکرار** شود.
6. **اعتبارسنجی:** وضعیت فعلی شعبه را شبیه‌سازی کنید و نتیجه را با داده‌ی واقعی مقایسه کنید. شبیه‌ساز تا چه حد واقعیت را بازتولید می‌کند؟ اختلاف‌ها از کجا می‌آیند؟
7. **پرسش‌های مدیر:**
   - **الف)** هر سه سیاست را با ۲، ۳ و ۴ باجه مقایسه کنید (دست‌کم ۲۰۰ روز شبیه‌سازی برای هر حالت) و نتیجه را در یک جدول بیاورید.
   - **ب)** هدف مدیر این است که **حداکثر ۵٪** مشتری‌ها بیش از ۱۵ دقیقه منتظر بمانند. کمترین تعداد باجه و بهترین سیاست برای رسیدن به این هدف چیست؟
   - **ج)** آیا به جای استخدام باجه‌دار چهارم برای کل روز، یک باجه‌دار **پاره‌وقت** فقط از ساعت <code dir="ltr">10:00</code> تا <code dir="ltr">12:00</code> کافی است؟ (شبیه‌ساز خود را طوری گسترش دهید که بتواند این حالت را شبیه‌سازی کند.)
   - **د)** ماه آینده یکی از باجه‌دارها یک ماه مرخصی است و شعبه با ۲ باجه کار می‌کند. کدام سیاست صف را پیشنهاد می‌کنید؟ پاسخ را فقط بر اساس میانگین انتظار ندهید؛ ببینید هر سیاست برای **کدام مشتری‌ها** بهتر یا بدتر است.
8. **پایداری و خروجی:** نشان دهید نتیجه‌ها به <code dir="ltr">seed</code> حساس‌اند یا نه (مثلاً با ۵ seed متفاوت). همه‌ی نتیجه‌های الزام ۷ را در یک فایل <code dir="ltr">results.json</code> و جدول خلاصه را در یک فایل CSV ذخیره کنید.

</div>

<div dir="rtl">

## ۴. محدودیت‌ها

- فقط پایتون و کتابخانه‌ی استاندارد: <code dir="ltr">csv</code>، <code dir="ltr">json</code>، <code dir="ltr">pathlib</code>، <code dir="ltr">datetime</code>، <code dir="ltr">random</code>، <code dir="ltr">math</code>، <code dir="ltr">statistics</code> و <code dir="ltr">collections</code>. استفاده از <code dir="ltr">numpy</code>، <code dir="ltr">pandas</code>، <code dir="ltr">matplotlib</code> یا هر کتابخانه‌ی بیرونی دیگر **مجاز نیست** (نمودار لازم نیست؛ اگر خواستید، نمودار متنی با <code dir="ltr">#</code> بکشید).
- فایل <code dir="ltr">bank&#95;log.csv</code> را دستی تغییر ندهید.
- همه‌ی اعداد گزارش باید از اجرای همان نوت‌بوکی بیایند که تحویل می‌دهید و با اجرای دوباره‌ی آن (از بالا به پایین) عیناً تکرار شوند؛ برای هر شبیه‌سازی <code dir="ltr">seed</code> را مشخص کنید.
- پروژه **انفرادی** است.

</div>

<div dir="rtl">

## ۵. آنچه تحویل می‌دهید

مهلت تحویل و زمان دفاع در کلاس اعلام می‌شود. دو فایل تحویل دهید:

**الف) نوت‌بوک کد** با نام <code dir="ltr">S20&#95;Project2&#95;StudentID.ipynb</code> (شماره‌ی دانشجویی خود را به جای <code dir="ltr">StudentID</code> بگذارید)؛ اجراشده، با خروجی همه‌ی سلول‌ها، و با توضیح کوتاه برای هر بخش. فایل‌های <code dir="ltr">results.json</code> و CSV خلاصه را هم ضمیمه کنید.

**ب) گزارش** با نام <code dir="ltr">S20&#95;Project2&#95;StudentID.pdf</code>، به فارسی، حداکثر ۶ صفحه، با این بخش‌ها:

1. **خلاصه‌ی مدیریتی** (حداکثر نیم صفحه): مهم‌ترین یافته‌ها و پیشنهادها به زبان ساده.
2. **داده و پاک‌سازی:** اندازه‌ی داده، انواع ایرادها، تعداد هر کدام و تصمیم شما درباره‌ی هر نوع.
3. **وضعیت فعلی شعبه:** آمار انتظار و ورود، باجه‌ها و موارد غیرعادی.
4. **شبیه‌ساز:** طراحی کلاس‌ها (یک نمودار ساده‌ی کلاس‌ها کافی است)، فرض‌های ساده‌کننده، و نتیجه‌ی اعتبارسنجی.
5. **پرسش‌های مدیر:** جدول مقایسه‌ی سیاست‌ها و پاسخ مستدل به هر پرسش.
6. **محدودیت‌ها:** چه چیزهایی را این داده و این شبیه‌ساز **نمی‌تواند** نشان دهد.
7. **پیوست:** فهرست سطرهای کنارگذاشته.

گزارش باید **خودبسنده** باشد: خواننده بدون باز کردن نوت‌بوک بفهمد چه کرده‌اید و به چه نتیجه‌ای رسیده‌اید. کد را در گزارش نیاورید.

</div>

<div dir="rtl">

## ۶. دفاع

جلسه‌ی دفاع حدود ۱۰ دقیقه است:

- ارائه‌ی کوتاه یافته‌ها (حدود ۳ دقیقه، از روی گزارش)،
- اجرای <code dir="ltr">clean&#95;log</code>، <code dir="ltr">fit&#95;model</code> و <code dir="ltr">simulate</code> شما با داده یا پارامترهای جدید،
- توضیح طراحی کلاس‌ها و بخش‌هایی از کد که پرسیده می‌شود (**هر خط** کد باید برای شما قابل توضیح باشد)،
- اعمال یک تغییر کوچک در همان جلسه (مثلاً عوض کردن ساعت بسته شدن شعبه یا افزودن یک سیاست ساده).

نمره‌ی هر بخش فقط در صورتی داده می‌شود که بتوانید در دفاع از آن بربیایید.

</div>

<div dir="rtl">

## ۷. معیارهای ارزیابی

| معیار | سهم |
|---|---|
| پاک‌سازی داده و مستندسازی تصمیم‌ها (الزام ۱) | ۱۵٪ |
| تحلیل وضعیت فعلی و موارد غیرعادی (الزامات ۲ و ۳) | ۱۵٪ |
| طراحی و درستی شبیه‌ساز: کلاس‌ها، وراثت، تکرارپذیری (الزامات ۴ و ۵) | ۲۵٪ |
| اعتبارسنجی، آزمایش‌ها و پاسخ به پرسش‌های مدیر (الزامات ۶ تا ۸) | ۱۵٪ |
| کیفیت گزارش: شفافیت، ساختار، دقت در اعداد | ۲۰٪ |
| خوانایی کد: نام‌گذاری، تقسیم به کلاس‌ها و توابع، توضیح‌ها | ۱۰٪ |
| **امتیازی:** مدل کردن مشتری‌هایی که بعد از انتظار طولانی شعبه را ترک می‌کنند، یا یک سیاست تازه (مثلاً نوبت اینترنتی یا اولویت برای افراد سالمند) که اثرش را با شبیه‌ساز نشان دهید | تا ۱۰٪ اضافه |

</div>